## Finding Similar structures with different melting points:

In [2]:
import os
import rootutils

from tqdm.notebook import tqdm

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
from tqdm.auto import tqdm
import numpy as np

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.DataStructs import TanimotoSimilarity, BulkTanimotoSimilarity

# Some docs here: https://www.rdkit.org/docs/GettingStartedInPython.html

In [4]:
df = pd.read_csv("data_cod/coord_numbs_temp_smiles.csv")

### Remove bugged smiles (Tanimoto Dist breaks on them):
- [N-]=[N+]=N[P](N=[N+]=[N-])(N=[N+]=[N-])(N=[N+]=[N-])(N=[N+]=[N-])N=[N+]=[N-].c1ccc(P(=N[P+](c2ccccc2)(c2ccccc2)c2ccccc2)(c2ccccc2)c2ccccc2)cc1

In [5]:
df = df[~df["can_smiles"].isin(["[N-]=[N+]=N[P](N=[N+]=[N-])(N=[N+]=[N-])(N=[N+]=[N-])(N=[N+]=[N-])N=[N+]=[N-].c1ccc(P(=N[P+](c2ccccc2)(c2ccccc2)c2ccccc2)(c2ccccc2)c2ccccc2)cc1"])]

---

## Compute TanimotoDistance Matrix:

In [6]:
def compute_morgan_fps(smiles: str, radius: int = 2, nbits: int = 2048):
    try:
        mol = Chem.MolFromSmiles(smiles)
        mfpgen = Chem.rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nbits)
        
        return mfpgen.GetFingerprint(mol)
    except Exception as e:
        with open('error_smiles.txt', 'a') as f:
            f.write(f"Error in fingerprints: {smiles}\n{str(e)}\n")
        return None

In [7]:
smiles_list = df["can_smiles"].tolist()

fps = [compute_morgan_fps(smiles) for smiles in tqdm(smiles_list, desc="Computing fingerprints")]

Computing fingerprints:   0%|          | 0/5586 [00:00<?, ?it/s]

In [8]:
def compute_similarity_matrix(fps):
    n = len(fps)
    M = np.zeros((n, n), dtype=float)
    for i in tqdm(range(n), desc="Computing Tanimoto matrix"):
        # compute similarities of fps[i] against fps[i+1:]
        sims = BulkTanimotoSimilarity(fps[i], fps[i+1:])
        # fill upper triangle and mirror to lower
        M[i, i+1:] = sims
        M[i+1:, i] = sims
        M[i, i] = 1.0  # similarity with self
    return M

In [9]:
sim_matrix = compute_similarity_matrix(fps)

Computing Tanimoto matrix:   0%|          | 0/5586 [00:00<?, ?it/s]

In [10]:
dist_matrix = 1 - sim_matrix

---
## Finding outlier pairs:

In [11]:
def find_close_pairs(
    df: pd.DataFrame,
    distance_matrix: np.ndarray,
    dist_threshold: float = 0.1,
    temp_threshold: float = 10,
):
    N = len(df)
    
    # Input validation
    assert "T" in df.columns, "DataFrame must have a 'T' column"
    assert isinstance(distance_matrix, np.ndarray) and distance_matrix.shape == (N, N), \
           "Distance matrix must be a NumPy array of shape (N, N)"
    
    # Get indices of the upper triangle (excluding diagonal)
    i, j = np.triu_indices(N, k=1)
    
    # Extract distances for these pairs
    distances = distance_matrix[i, j]
    
    # Get temperatures as a NumPy array for efficient indexing
    T = df["T"].values
    T_i = T[i]
    T_j = T[j]
    
    # Filter pairs where distance is less than the threshold
    mask_dist = (distances < dist_threshold) & (distances > 0)
    
    selected_i = i[mask_dist]
    selected_j = j[mask_dist]
    selected_T_i = T_i[selected_i]
    selected_T_j = T_j[selected_j]
    selected_distances = distances[mask_dist]

    # Create a DataFrame with the results
    result_df = pd.DataFrame({
        'id_i': df["id"].values[selected_i],
        'id_j': df["id"].values[selected_j],
        'smiles_i': df["can_smiles"].values[selected_i],
        'smiles_j': df["can_smiles"].values[selected_j],
        'i': selected_i,
        'j': selected_j,
        'T_i': selected_T_i,
        'T_j': selected_T_j,
        'Tanimoto_dist': selected_distances,
        'T_diff': np.abs(selected_T_i - selected_T_j),
    })
    
    result_df = result_df[result_df["T_diff"] > temp_threshold]
    result_df.sort_values(by=["T_diff"], inplace=True, ascending=False)
    
    return result_df

In [15]:
close_pairs_df = find_close_pairs(df, dist_matrix, dist_threshold=0.1, temp_threshold=20)

In [16]:
close_pairs_df

,id_i,id_j,smiles_i,smiles_j,i,j,T_i,T_j,Tanimoto_dist,T_diff
449,7118368,7118374,CCOC(=O)C[C@@H]1c2cc(Br)ccc2OC(=O)[C@H]1[C@H]1...,CCOC(=O)C[C@@H]1c2cc(Br)ccc2OC(=O)[C@H]1[C@@H]...,4153,4158,180.0,-189.0,0.053571,369.0
65,2006810,2006811,O=C1CC(C(=O)O)c2ccccc21,O.O=C1CC(C(=O)O)c2ccccc21,648,649,180.0,-178.0,0.037037,358.0
17,1513327,2007623,OC1CCCC1,OC1CCCCCCCCC1,175,700,180.0,-116.0,0.090909,296.0
59,2000642,2102056,O=C(c1ccccc1)c1ccc(C(=O)c2ccccc2)cc1,O=C(c1ccccc1)c1ccccc1,460,1411,180.0,-112.0,0.066667,292.0
438,7100087,7200402,O=C(OCCCCOC(=O)[C@H](O)C(F)(F)F)[C@@H](O)C(F)(F)F,O=C(OCCCCCCCCCCOC(=O)[C@H](O)C(F)(F)F)[C@H](O)...,3983,4430,180.0,-110.4,0.045455,290.4
...,...,...,...,...,...,...,...,...,...,...
278,4508011,4508038,O=C(O)CCCCC(=O)O,O=C(O)CCCCCCCCC(=O)O,3717,3739,180.0,202.0,0.083333,22.0
307,4508012,4508038,O=C(O)CCCCC(=O)O,O=C(O)CCCCCCCCC(=O)O,3718,3739,180.0,202.0,0.083333,22.0
187,4116143,4508038,O=C(O)CCCCC(=O)O,O=C(O)CCCCCCCCC(=O)O,3503,3739,180.0,202.0,0.083333,22.0
148,2221191,4508038,O=C(O)CCCCC(=O)O,O=C(O)CCCCCCCCC(=O)O,2583,3739,180.0,202.0,0.083333,22.0


In [14]:
close_pairs_df.to_csv("close_pairs.csv", index=False)